[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

# 📝 Ejercicio Guiado — Predicción de Precios de Teléfonos Móviles con Ensambles

**Nombre:** ___________________________
**Fecha:** ___________________________

---

En este ejercicio construirás modelos de **Random Forest Regressor** y **Gradient Boosting Regressor** para predecir el precio (en USD) de teléfonos móviles a partir de sus especificaciones técnicas.

El dataset proviene de Kaggle:
📦 [Mobile Phone Price Dataset](https://www.kaggle.com/datasets/rkiattisak/mobile-phone-price)

Seguirás una estructura parecida al notebook de referencia de clasificación (**Penguins RF & GB**) para construir tus `Pipeline` y `ColumnTransformer`, pero adaptada a un problema de **regresión** con un dataset bastante más "sucio":

| Aspecto | Penguins RF & GB (referencia) | **Este ejercicio** |
|---|---|---|
| Tarea | Clasificación | **Regresión** |
| División | Train / Test | **Train / Validation / Test** |
| Preprocesamiento numérico | `StandardScaler` | `StandardScaler` (dentro de un `Pipeline` con imputación) |
| Preprocesamiento categórico | `OneHotEncoder` | `OneHotEncoder` (dentro de un `Pipeline` con imputación) |
| Limpieza de datos | Mínima (eliminar nulos) | **Importante** — columnas numéricas guardadas como texto, formatos irregulares |
| Búsqueda de hiperparámetros | `GridSearchCV` | `GridSearchCV` (con métrica de regresión) |
| Evaluación | Accuracy, F1, matriz de confusión | **R², MAE (con interpretación obligatoria)** |
| Predicción final | — | **Muestra sintética propia** |

### ⚠️ Los retos de este dataset

A diferencia de Penguins (donde las columnas ya venían limpias), aquí vas a tener que **tomar decisiones de preprocesamiento por tu cuenta**. Dos columnas en particular te van a dar trabajo:

- **`Camera (MP)`**: no es un número simple. Puede tener uno o varios valores (múltiples lentes) y a veces texto que no es una resolución.
- **`Price ($)`** (¡tu variable objetivo!): viene mezclada con símbolos de moneda, separadores de miles y espacios.

No te vamos a dar la línea de código exacta para limpiarlas — te daremos pistas y preguntas guía. **Tú decides qué camino seguir.**

### Columnas del dataset

| Columna | Tipo | Descripción |
|---|---|---|
| `Brand` | Categórica | Marca del fabricante (Apple, Samsung, Xiaomi...) |
| `Model` | Categórica ⚠️ alta cardinalidad | Modelo específico — casi todos los valores son únicos |
| `Storage` | Numérica/Texto | Almacenamiento interno (ej. `"128 GB"`) |
| `RAM` | Numérica/Texto | Memoria RAM (ej. `"6 GB"`) |
| `Screen Size (inches)` | Numérica | Tamaño de pantalla en pulgadas |
| `Camera (MP)` | Texto ⚠️ **reto** | Especificación de cámara(s), formato irregular (ej. `"48 + 8 + 2 + 2"`) |
| `Battery Capacity (mAh)` | Numérica | Capacidad de batería |
| `Price ($)` 🎯 ⚠️ **reto** | Numérica/Texto | **Variable objetivo** — precio en USD, formato irregular |

> 💡 Los nombres exactos de columnas (espacios extra, mayúsculas) pueden variar levemente según la versión del CSV que descargues. Verifica siempre con `df.columns.tolist()` antes de escribir código que dependa de un nombre exacto.

## 0. Descarga del dataset

**Opción A – Kaggle API:**
```bash
!pip install kaggle -q
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d rkiattisak/mobile-phone-price --unzip
```

**Opción B – Descarga manual:**
Descarga el archivo `.csv` desde Kaggle y súbelo a la misma carpeta del notebook.

> ⚠️ Revisa el nombre exacto del archivo descargado (puede ser algo como `Mobile phone price.csv`) y ajústalo en el `pd.read_csv(...)` de la siguiente sección.

## 1. Librerías

Importa todas las librerías que necesitarás durante el ejercicio.

**Pistas — qué importar y de dónde:**
- `pandas`, `numpy`, `re` (expresiones regulares) y `matplotlib.pyplot`
- De `sklearn.ensemble`: `RandomForestRegressor` y `GradientBoostingRegressor`
- De `sklearn.model_selection`: `train_test_split` y `GridSearchCV`
- De `sklearn.pipeline`: `Pipeline`
- De `sklearn.compose`: `ColumnTransformer`
- De `sklearn.preprocessing`: `StandardScaler`, `OneHotEncoder`
- De `sklearn.impute`: `SimpleImputer`
- De `sklearn.metrics`: `mean_absolute_error`, `mean_squared_error`, `r2_score`
- Define `random_state = 42` para reproducibilidad

> 💡 **¿Por qué `re` (regex)?**
> Vas a necesitar extraer números de texto irregular (por ejemplo, sacar `128` de `"128 GB"`, o los distintos valores de MP de `"48 + 8 + 2 + 2"`). Las expresiones regulares son la herramienta ideal para esto. Si no las conoces, la función `re.findall(r'\d+\.?\d*', texto)` te devuelve una lista con todos los números que aparecen en un string — puede servirte como punto de partida.

In [ ]:
# Importa aquí todas las librerías necesarias:


# Define random_state para reproducibilidad:
random_state = 42

## 2. Carga y limpieza de datos

Carga el dataset y realiza las transformaciones necesarias para poder usarlo. Esta es la sección más importante del ejercicio — tómate tu tiempo.

**Pistas generales:**
- Usa `pd.read_csv('nombre_del_archivo.csv')` para cargar
- Revisa `.shape`, `.head()`, `.info()` e `.isnull().sum()`
- Verifica los nombres de columnas con `df.columns.tolist()`. Si encuentras espacios extra al inicio o final de los nombres, límpialos con `df.columns = df.columns.str.strip()`

### 2a. `Storage` y `RAM`
Estas columnas vienen como texto (ej. `"128 GB"`, `"6 GB"`). Necesitas quedarte solo con el número y convertirlo a tipo numérico.
- Revisa los valores únicos con `.unique()` antes de limpiar, para confirmar el patrón exacto (¿siempre dice "GB"? ¿hay espacios inconsistentes?)
- Una combinación de `.str.extract()` o `.str.replace()` con expresiones regulares, seguida de `.astype(int)`, debería funcionar

### 2b. `Camera (MP)` — ⚠️ reto
Esta columna **no es un solo número**. Antes de programar nada, explora:
```python
df['Camera (MP)'].sample(15)
```
Vas a notar que:
- Puede haber **uno o varios valores separados por `+`** (varios lentes: cámara principal, ultra gran angular, macro, profundidad...)
- Algunos de esos valores **no son una resolución en MP** (por ejemplo, sensores marcados como `"3D"` o `"ToF"` que no tienen megapíxeles)

**Preguntas para guiar tu decisión:**
- ¿Qué información resumen quieres extraer de esta columna? Algunas opciones válidas: número de cámaras (lentes), suma total de MP, el valor máximo de MP (cámara principal), el promedio.
- ¿Puedes crear más de una columna nueva a partir de esta? (por ejemplo `n_camaras` y `camera_mp_total`)
- ¿Cómo vas a manejar los tokens que no son números (`"3D"`, `"ToF"`)? Si los ignoras, ¿pierdes información importante o es razonable descartarlos?

No hay una única respuesta correcta — documenta con un comentario **por qué** elegiste tu estrategia.

### 2c. `Price ($)` — ⚠️ reto (¡es tu variable objetivo!)
Explora los valores únicos o una muestra:
```python
df['Price ($)'].sample(15)
```
Es muy probable que encuentres una mezcla de formatos: algunos precios con el símbolo `$`, otros sin él, algunos con comas como separador de miles (ej. `"$1,199"`), y posiblemente espacios en blanco sobrantes.

**Preguntas para guiar tu decisión:**
- ¿Qué caracteres necesitas eliminar antes de poder convertir la columna a numérica?
- Si intentas `pd.to_numeric(df['Price ($)'])` directamente sin limpiar, ¿qué error obtienes? ¿Qué te dice ese error sobre los datos?
- Después de limpiar, verifica que no haya quedado ningún valor nulo o mal convertido en tu variable objetivo — un solo precio corrupto puede arruinar el entrenamiento.
- Considera renombrar la columna limpia a algo más simple como `Price` para que sea más fácil de referenciar en el resto del notebook.

### 2d. `Model` — decisión de diseño
Revisa cuántos valores únicos tiene esta columna con `.nunique()` y compáralo con el número total de filas.
- ¿Tiene sentido usar esta columna como feature categórica? Piensa en qué pasaría si aplicaras `OneHotEncoder` sobre una columna donde casi todos los valores son únicos — ¿el modelo podría generalizar a modelos de teléfono que nunca vio en entrenamiento?
- Decide si la vas a **descartar** o si vas a intentar extraer algo útil de ella (por ejemplo, si el nombre del modelo contiene palabras como "Pro", "Max", "Lite"). Justifica tu decisión con un comentario.

In [ ]:
# Carga el dataset:


In [ ]:
# Revisa tipos de datos, dimensiones, nombres de columnas y valores nulos:


In [ ]:
# Limpia las columnas Storage y RAM (elimina el sufijo textual y convierte a numérico):


In [ ]:
# 2b. Limpia / transforma la columna Camera (MP).
# Explora primero los valores únicos, luego decide tu estrategia (ver pistas arriba).
# Documenta tu decisión con un comentario antes de tu código:



In [ ]:
# 2c. Limpia la columna Price ($) y conviértela a numérica.
# Recuerda verificar que no queden nulos después de la limpieza.



In [ ]:
# 2d. Decide qué hacer con la columna Model (descartarla o transformarla) y justifica con un comentario:



In [ ]:
# Verifica que todos los tipos de dato sean correctos después de la limpieza (.dtypes):


## 3. Análisis Exploratorio (EDA)

Antes de modelar, es fundamental entender los datos. Realiza los siguientes análisis y escribe una interpretación debajo de cada gráfica.

### 3a. Distribución del target (`Price`)
**Pistas:**
- Crea un histograma con `plt.hist(df['Price'], bins=40, ...)`
- ¿La distribución es simétrica o tiene sesgo? ¿Hay outliers evidentes (teléfonos ultra premium)?
- Calcula también la media, mediana y desviación estándar del precio

### 3b. Correlación de variables numéricas con el target
**Pistas:**
- Selecciona solo las columnas numéricas con `.select_dtypes(include='number')` (deben incluir tus columnas nuevas derivadas de `Camera (MP)`)
- Calcula la correlación con `.corr()['Price'].drop('Price').sort_values()`
- Grafica las correlaciones como barras horizontales
- ¿Qué variables numéricas están más correlacionadas con el precio? ¿Tus features de cámara aportan señal?

### 3c. Boxplots de variables categóricas / discretas clave
**Pistas:**
- Crea boxplots de `Price` agrupados por `Brand`
- Crea también un boxplot de `Price` agrupado por `RAM` (una vez numérica, ordénala antes de graficar)
- Usa `df.boxplot(column='Price', by='Brand', ...)` o matplotlib manual
- ¿Qué marca tiene precios más altos? ¿La relación con RAM es la que esperarías?

In [ ]:
# 3a. Histograma del target (Price):
# Pista: usa plt.hist(..., bins=40, edgecolor='white')
# Agrega líneas verticales para la media (roja) y la mediana (verde)
# Pista: plt.axvline(df['Price'].mean(), color='red', linestyle='--', label='Media')



**📝 Interpretación del histograma:**
*Escribe aquí tu análisis. Responde: ¿la distribución es simétrica o tiene sesgo? ¿La media o la mediana es más representativa? ¿Hay teléfonos extremadamente caros que podrían ser outliers?*

*(Tu respuesta aquí...)*

In [ ]:
# 3b. Correlación de variables numéricas con Price:
# Pista:
#   num_df = df.select_dtypes(include='number')
#   corr = num_df.corr()['Price'].drop('Price').sort_values()
#   plt.barh(corr.index, corr.values, color=[...])



**📝 Interpretación del gráfico de correlaciones:**
*¿Qué variable numérica tiene mayor correlación positiva con el precio? ¿Tiene sentido intuitivo? ¿Tus features derivadas de `Camera (MP)` resultaron útiles o sorpresivamente irrelevantes?*

*(Tu respuesta aquí...)*

In [ ]:
# 3c. Boxplot de Price por Brand:
# Pista: df.boxplot(column='Price', by='Brand', figsize=(12,5), rot=60)
# Ajusta el título con plt.title(...) y elimina el título automático con plt.suptitle('')



**📝 Interpretación del boxplot por Brand:**
*¿Qué marca tiene los precios más elevados en mediana? ¿Cuál tiene mayor dispersión? ¿Hay outliers notables en alguna marca?*

*(Tu respuesta aquí...)*

In [ ]:
# 3c. Boxplot de Price por RAM (ordena los valores numéricamente antes de graficar):
# Pista: df_sorted = df.copy(); df_sorted['RAM'] = ...



**📝 Interpretación del boxplot por RAM:**
*¿A más RAM, mayor precio? ¿La relación es lineal o hay saltos abruptos? ¿Qué implicación tiene esto para el modelo?*

*(Tu respuesta aquí...)*

## 4. División de datos: Train / Validation / Test

Dividirás los datos en **tres conjuntos**:

| Conjunto | Proporción | Uso |
|---|---|---|
| **Train** | 60% | Entrenamiento y GridSearchCV |
| **Validation** | 20% | Comparar modelos y elegir el mejor |
| **Test** | 20% | Evaluación final — ¡solo tocar al final! |

**Pistas:**
- Primero separa la variable objetivo (`Price`) de las features
- Haz un primer `train_test_split` para separar test (20%) del resto (80%)
- Luego haz un segundo `train_test_split` sobre el 80% para obtener train (75% del 80% = 60% total) y validation (25% del 80% = 20% total)
- Usa `random_state=random_state` en ambas divisiones
- Imprime el tamaño de cada conjunto para verificar las proporciones

> 💡 **¿Por qué tres conjuntos?**
> Si usas el test set para elegir entre modelos, contaminas la evaluación final.
> El validation set te permite comparar modelos honestamente sin «quemar» el test set.

In [ ]:
# Separa X e y:
TARGET = 'Price'
# X = ...
# y = ...


# Primera división: separa el test set (20%):
# X_temp, X_test, y_temp, y_test = train_test_split(..., test_size=0.2, random_state=random_state)


# Segunda división: del 80% restante, separa validation (25% = 20% del total):
# X_train, X_val, y_train, y_val = train_test_split(..., test_size=0.25, random_state=random_state)


# Verifica los tamaños:


## 5. Identificación de columnas por tipo

Identifica qué columnas son numéricas y cuáles son categóricas para aplicar transformaciones distintas.

**Pistas:**
- Usa `X_train.select_dtypes(include='number').columns` para numéricas
- Usa `X_train.select_dtypes(include='object').columns` para categóricas
- Imprime ambas listas para verificar
- Recuerda tu decisión sobre `Model` (sección 2d): si decidiste descartarla, no debería aparecer aquí

> ⚠️ Si tu columna categórica `Brand` (o cualquier otra que conserves) tiene muchos valores únicos, piensa si `OneHotEncoder` sigue siendo una buena idea, igual que reflexionaste con `Model`.

In [ ]:
# Identifica las columnas numéricas:


# Identifica las columnas categóricas:


# Revisa cuántos valores únicos tiene cada columna categórica:
# Pista: X_train[cat_cols].nunique()


## 6. Preprocesamiento con ColumnTransformer

Construye un `ColumnTransformer` con las transformaciones apropiadas para cada tipo de columna, siguiendo el mismo patrón usado en el notebook de referencia (Penguins RF & GB): un `Pipeline` numérico y un `Pipeline` categórico combinados en un `ColumnTransformer`.

### Pipeline numérico
1. `SimpleImputer(strategy='median')` — imputa nulos con la mediana (por si algún valor no se pudo limpiar/convertir correctamente)
2. `StandardScaler()` — estandariza las variables

### Pipeline categórico
1. `SimpleImputer(strategy='most_frequent')`
2. `OneHotEncoder(handle_unknown='ignore')`

**Pistas:**
- Combina ambos pipelines en un `ColumnTransformer`, igual que en Penguins:
```python
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ]
)
```
- `num_cols` y `cat_cols` son las listas que definiste en la sección anterior

In [ ]:
# Crea el pipeline numérico (SimpleImputer -> StandardScaler):


# Crea el pipeline categórico (SimpleImputer -> OneHotEncoder):


# Combina ambos en un ColumnTransformer:



## 7. Definición de modelos y búsqueda de hiperparámetros

Define los dos pipelines (RF y GB) y la grilla de hiperparámetros para `GridSearchCV`.

**Pistas para los pipelines:**
```python
pipeline_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=random_state))
])
```

**Pistas para la grilla de hiperparámetros:**
Nota que el prefijo es `regressor__` (no `classifier__` como en el notebook de referencia):
```python
param_grid = {
    'regressor__n_estimators': [50, 100],
    'regressor__max_depth': [3, 5, 10, None],
    'regressor__min_samples_leaf': [1, 5, 10]
}
```

**Pistas para GridSearchCV:**
- Usa `cv=3` para validación cruzada de 3 folds
- Usa `scoring='neg_mean_absolute_error'` (negativo porque GridSearchCV maximiza)
- Usa `n_jobs=-1` para paralelizar y acelerar la búsqueda
- Crea un `GridSearchCV` para el RF y otro para el GB

> 💡 **¿Por qué `neg_mean_absolute_error`?**
> `GridSearchCV` siempre **maximiza** el score. Como el MAE es un error que queremos minimizar, se usa el valor negativo: maximizar `-MAE` equivale a minimizar `MAE`.

In [ ]:
# Define el pipeline para Random Forest:


# Define el pipeline para Gradient Boosting:


# Define la grilla de hiperparámetros:


# Define GridSearchCV para Random Forest:


# Define GridSearchCV para Gradient Boosting:



## 8. Entrenamiento

Entrena ambos modelos sobre los datos de entrenamiento. Puede tardar unos minutos.

**Pistas:**
- Usa `rf_search.fit(X_train, y_train)` y `gb_search.fit(X_train, y_train)`
- Después del entrenamiento, muestra los mejores hiperparámetros con `.best_params_`
- Muestra también el mejor score de CV con `.best_score_` (recuerda que va a ser negativo por `neg_mean_absolute_error`)

> ⚠️ Si el entrenamiento tarda demasiado, reduce la grilla a menos combinaciones (por ejemplo, solo 2 valores por hiperparámetro).

In [ ]:
# Entrena el Random Forest (GridSearchCV):


# Entrena el Gradient Boosting (GridSearchCV):



In [ ]:
# Muestra los mejores hiperparámetros encontrados para Random Forest:


# Muestra los mejores hiperparámetros encontrados para Gradient Boosting:



## 9. Evaluación de métricas — R² y MAE

Evalúa ambos modelos en los tres conjuntos de datos y **compara e interpreta** sus resultados. Esta sección es central en el ejercicio: no basta con calcular los números, tienes que explicar qué significan.

**Métricas a calcular para cada modelo y cada conjunto (train, val, test):**
- **MAE** (`mean_absolute_error`) — error promedio absoluto, **en dólares**. Un MAE de 120 significa que, en promedio, la predicción se equivoca por $120.
- **R²** (`r2_score`) — proporción de la varianza del precio que el modelo logra explicar (1.0 = perfecto, 0.0 = igual que predecir siempre la media).
- (Opcional) **RMSE** — penaliza más los errores grandes que el MAE.

**Pistas:**
- Crea una función `evaluar(nombre, modelo, X, y)` que reciba el modelo y los datos y retorne (o imprima) MAE y R² (y RMSE si quieres)
- Llama a la función para (RF en train), (RF en val), (GB en train), (GB en val)
- **Solo al final**, evalúa en test el modelo que mejor haya funcionado en validación
- Organiza los resultados en una tabla con `pd.DataFrame`

**Preguntas que debes responder con tus resultados (no solo reportar los números):**
- Tu modelo obtuvo un MAE de test de $X. Mirando el rango de precios del dataset (mínimo, máximo, mediana calculados en el EDA), ¿ese error te parece grande o pequeño?
- Tu modelo obtuvo un R² de test de Y. ¿Qué porcentaje de la variabilidad del precio logra explicar el modelo? ¿Qué tan cerca está de 1.0?
- ¿El R² y el MAE cuentan la misma historia, o uno es más optimista que el otro? ¿Por qué podrían diferir?

> 💡 **Señales de sobreajuste:**
> Si el R² en train es 0.98 pero en val es 0.65 → el modelo memorizó el train.
> Un buen modelo tiene métricas similares en train, val y test.

In [ ]:
# Crea una función que calcule MAE y R² (y opcionalmente RMSE) dado un modelo y un conjunto de datos:
# def evaluar(nombre, modelo, X, y):
#     y_pred = modelo.predict(X)
#     ...



In [ ]:
# Evalúa ambos modelos en train y validación:



In [ ]:
# Elige el mejor modelo según val, y evalúalo en el test set:
# ⚠️ Solo hacer esto una vez al final — no uses test para tomar decisiones



**📝 Interpretación de MAE y R² (obligatorio):**
*Responde aquí con tus números reales: ¿cuál es el MAE de test en dólares y qué tan grande es comparado con el precio típico de un teléfono en este dataset? ¿Cuál es el R² de test y qué fracción de la variabilidad del precio explica tu modelo? ¿Consideras que el modelo es útil en la práctica?*

*(Tu respuesta aquí...)*

## 10. Visualización de resultados

Genera las siguientes gráficas para el **mejor modelo en el test set**:

**Gráfica 1 — Real vs Predicho:**
- Scatter plot de `y_test` (eje x) vs `y_pred_test` (eje y)
- Agrega la línea diagonal de predicción perfecta en rojo
- Etiqueta los ejes y agrega el R² en el título

**Gráfica 2 — Distribución de residuos:**
- Histograma de `y_test - y_pred_test`
- Línea vertical en 0
- ¿Los residuos son simétricos alrededor de 0? ¿Hay una cola de errores grandes en teléfonos caros?

**Gráfica 3 — Importancia de features:**
- Accede a las importancias con `modelo.best_estimator_.named_steps['regressor'].feature_importances_`
- Muéstralas como barras horizontales ordenadas de mayor a menor
- Para los nombres de features, recuerda que el `ColumnTransformer` reorganizó las columnas

> 💡 **Pista para nombres de features con ColumnTransformer:**
> Usa `preprocessor.get_feature_names_out()` después de hacer `.fit()` para obtener los nombres en el mismo orden que las importancias.

In [ ]:
# Gráfica 1: Real vs Predicho en test set:



In [ ]:
# Gráfica 2: Distribución de residuos en test set:



In [ ]:
# Gráfica 3: Importancia de features del mejor modelo:
# Pista: feature_importances_ está en el paso 'regressor' del pipeline
# best_model.best_estimator_.named_steps['regressor'].feature_importances_



## 11. Reflexión sobre los resultados

Responde estas preguntas aquí (doble clic para editar):

**1. ¿Qué modelo (RF o GB) funcionó mejor en validación? ¿Y en test? ¿Coinciden?**
*(Tu respuesta aquí...)*

---

**2. Mirando la gráfica de importancia de features, ¿qué variable impacta más el precio? ¿Tus features derivadas de `Camera (MP)` quedaron entre las más importantes?**
*(Tu respuesta aquí...)*

---

**3. ¿Hay indicios de sobreajuste? ¿Cómo lo sabes comparando las métricas de train vs val?**
*(Tu respuesta aquí...)*

---

**4. Vuelve a tu estrategia de limpieza de `Camera (MP)` y `Price ($)` (sección 2). ¿Cambiarías algo ahora que ves los resultados? ¿Probaste alguna alternativa?**
*(Tu respuesta aquí...)*

---

**5. ¿Para qué sirve el conjunto de validación si ya usamos validación cruzada (GridSearchCV) dentro del train? ¿Son lo mismo?**
*(Tu respuesta aquí...)*

## 12. 🧪 Prueba con muestra sintética propia

Diseña **al menos 5 teléfonos ficticios** con perfiles muy diferentes y predice su precio.

La clave es razonar **antes de ejecutar** cuánto debería costar cada teléfono según sus especificaciones.

**Perfiles sugeridos:**

| # | Perfil | Características esperadas |
|---|---|---|
| 1 | Gama alta actual | Marca top, mucha RAM/almacenamiento, cámara múltiple de alta resolución |
| 2 | Cámara como diferencial | Muchos lentes / MP alto, gama media en lo demás |
| 3 | Gama baja | Poca RAM/almacenamiento, batería y cámara modestas |
| 4 | Batería extendida | Batería muy grande (mAh), resto gama media |
| 5 | Compacto económico | Pantalla pequeña, especificaciones básicas |

**Pistas importantes:**
- El DataFrame debe tener **exactamente las mismas columnas** que usó tu pipeline entrenado (recuerda si descartaste `Model` u otras columnas)
- Los valores de columnas categóricas (como `Brand`) deben coincidir con los que aparecen en el dataset original (usa `.unique()` para revisar valores válidos)
- Si creaste features derivadas de `Camera (MP)` (número de cámaras, MP total, etc.), debes calcular esos mismos valores para tus teléfonos ficticios, de forma consistente con la función/lógica que definiste en la sección 2b
- `Storage` y `RAM` deben ser numéricos (sin "GB") — ¡ya que limpiaste esas columnas!
- Usa `mejor_modelo.predict(muestra)` para obtener las predicciones

In [ ]:
# Revisa los valores válidos de las columnas categóricas para no cometer errores:
# Pista: for col in cat_cols: print(col, X_train[col].unique())



In [ ]:
# Diseña tus 5 teléfonos ficticios.
# ANTES de ejecutar, escribe en comentarios cuánto esperas que cueste cada uno:

# Teléfono 1 (perfil: ...): precio esperado ~$___
# Teléfono 2 (perfil: ...): precio esperado ~$___
# Teléfono 3 (perfil: ...): precio esperado ~$___
# Teléfono 4 (perfil: ...): precio esperado ~$___
# Teléfono 5 (perfil: ...): precio esperado ~$___

# Construye tu DataFrame con las mismas columnas (y features derivadas) que usó tu pipeline:
muestra = pd.DataFrame({
    # Completa con las columnas que tu pipeline final espera
})

muestra

In [ ]:
# Genera las predicciones de precio para tus teléfonos:
# Pista: predicciones = mejor_modelo.predict(muestra)


# Muestra los resultados en una tabla legible:
# Pista: muestra_result = muestra.copy()
#        muestra_result['Precio Predicho ($)'] = predicciones.round(2)



### Reflexión sobre la muestra sintética

Responde aquí (doble clic para editar):

**1. ¿Las predicciones se acercan a lo que esperabas? ¿Cuál te sorprendió más y por qué?**
*(Tu respuesta aquí...)*

---

**2. ¿El modelo diferencia bien entre un teléfono básico y uno de gama alta? ¿La diferencia de precio predicha te parece razonable?**
*(Tu respuesta aquí...)*

---

**3. Prueba cambiar solo la cámara (número de lentes o MP total) manteniendo todo lo demás igual. ¿Cuánto cambia el precio predicho? ¿Tiene sentido con lo que viste en la importancia de features?**
*(Tu respuesta aquí...)*